# ICLR Paper Language Analysis
Independent student project exploring associations between abstract language, author metrics, and Oral/Spotlight versus Poster presentation labels among accepted ICLR papers.

This is exploratory observational analysis, not causal inference or acceptance/rejection prediction. Run from the repository root after following README.md. Original analysis sections are retained; publication fixes are documented in CHANGELOG.md. Historical outputs and personal metadata are removed.


In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
import statsmodels.api as sm
from sklearn.neighbors import NearestNeighbors
from scipy.stats import skew, kurtosis

num_features = ['word_count', 'pos_score', 'neg_score', 'certainty',
                'achievement', 'promotional_ratio', 'max_h_index', 'avg_citations']
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Liberation Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
# Resources are installed explicitly; running this notebook never downloads data.
for resource in ['tokenizers/punkt_tab/english', 'sentiment/vader_lexicon.zip']:
    try:
        nltk.data.find(resource)
    except LookupError as exc:
        raise RuntimeError('Install NLTK resources first: python -m nltk.downloader punkt_tab vader_lexicon') from exc


In [ ]:
# 1. Load only the six analysis fields; identifiers and author names are excluded.
REQUIRED_COLUMNS = ['abstract', 'accept_type', 'max_h_index',
                    'avg_citations', 'author_count', 'year']

def load_dataset(path):
    frame = pd.read_csv(path, usecols=lambda name: name in REQUIRED_COLUMNS)
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    frame = frame[REQUIRED_COLUMNS].copy().reset_index(drop=True)
    if frame.empty or frame['abstract'].isna().any() or frame['accept_type'].isna().any():
        raise ValueError('A nonempty dataset with nonmissing abstracts and presentation labels is required.')
    frame['abstract'] = frame['abstract'].astype(str)
    frame['accept_type'] = frame['accept_type'].astype(str).str.lower().str.strip()
    if frame['abstract'].str.strip().eq('').any():
        raise ValueError('Empty abstracts are not supported.')
    allowed_labels = {'poster', 'oral', 'spotlight'}
    if not set(frame['accept_type']).issubset(allowed_labels):
        raise ValueError('Expected accepted-paper labels only: poster, oral, spotlight.')
    for column in ['max_h_index', 'avg_citations', 'author_count', 'year']:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
        if not np.isfinite(frame[column]).all() or (frame[column] < 0).any():
            raise ValueError(f'{column} must contain finite nonnegative numbers.')
    if (frame['author_count'] < 1).any() or (frame['author_count'] % 1 != 0).any():
        raise ValueError('author_count must contain positive integers.')
    if (frame['year'] % 1 != 0).any():
        raise ValueError('year must contain integers.')
    frame['year'] = frame['year'].astype(int)
    target = frame['accept_type'].isin(['oral', 'spotlight']).astype(int)
    if target.nunique() != 2 or target.value_counts().min() < 4:
        raise ValueError('Both presentation groups need at least four papers for a stratified holdout.')
    return frame

DATA_PATH = Path(os.environ.get('ICLR_DATA_PATH', 'data/iclr_cleaned_v1_20250322.csv'))
df = load_dataset(DATA_PATH)
print(f'Loaded {len(df)} accepted papers; raw identifiers and names were excluded.')


In [ ]:
# ====================== 2. 宣传词特征 ======================
promotional_words = ['novel','breakthrough','state-of-the-art','outperform','significant',
                     'superior','groundbreaking','innovative','unprecedented','remarkable',
                     'achieve','advance','improve','surpass','pioneer','revolutionary',
                     'best','first','new','effective','powerful','strong','excellent']

df['promotional_ratio'] = df['abstract'].apply(
    lambda text: sum(1 for w in str(text).split() if w.lower() in promotional_words) / len(str(text).split())
    if str(text).strip() else 0.0
)
print("宣传词特征提取完成 ✓")

In [ ]:
# 3. VADER sentiment and small handcrafted dictionaries (not official LIWC)
sia = SentimentIntensityAnalyzer()

def extract_linguistic_features(text):
    text = str(text)
    if not text.strip():
        return pd.Series({'word_count':0,'sent_count':0,'pos_score':0.0,'neg_score':0.0,'certainty':0,'achievement':0})
    sentences = nltk.sent_tokenize(text)
    words = nltk.word_tokenize(text)
    sentiment = sia.polarity_scores(text)
    certainty_words = ['certainly','definitely','clearly','obviously','undoubtedly','surely']
    achievement_words = ['achieve','success','outperform','surpass','advance','breakthrough']
    return pd.Series({
        'word_count': len(words),
        'sent_count': len(sentences),
        'pos_score': sentiment['pos'],
        'neg_score': sentiment['neg'],
        'certainty': sum(1 for w in words if w.lower() in certainty_words),
        'achievement': sum(1 for w in words if w.lower() in achievement_words)
    })

liwc_features = df['abstract'].apply(extract_linguistic_features)
df = pd.concat([df, liwc_features], axis=1)
print("Handcrafted linguistic features extracted.")

In [ ]:
# 4. Define TF-IDF; fitting is deferred until AFTER the train/test split.
tfidf = TfidfVectorizer(max_features=150, stop_words='english', min_df=2)


In [ ]:
# 5. Define numeric features; retain raw units in df for exploratory plots.
feature_cols = ['word_count','sent_count','pos_score','neg_score','certainty',
                'achievement','promotional_ratio','max_h_index','avg_citations','author_count']
scaler = StandardScaler()


In [ ]:
# 6. Presentation distinction among ACCEPTED papers, not acceptance/rejection.
df['is_high_level'] = df['accept_type'].isin(['spotlight', 'oral']).astype(int)


In [ ]:
# 7. One stratified holdout shared by both models. Fit preprocessing on training data only.
y = df['is_high_level']
train_idx, test_idx = train_test_split(df.index, test_size=0.25, random_state=42, stratify=y)
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
X_train_core = pd.DataFrame(scaler.fit_transform(df.loc[train_idx, feature_cols]),
                            columns=feature_cols, index=train_idx)
X_test_core = pd.DataFrame(scaler.transform(df.loc[test_idx, feature_cols]),
                           columns=feature_cols, index=test_idx)
train_tfidf = tfidf.fit_transform(df.loc[train_idx, 'abstract'])
test_tfidf = tfidf.transform(df.loc[test_idx, 'abstract'])
# The actual learned vocabulary may contain fewer than 150 terms.
tfidf_cols = [f'tfidf_{i}' for i in range(train_tfidf.shape[1])]
X_train_full = pd.concat([X_train_core, pd.DataFrame(train_tfidf.toarray(),
                         columns=tfidf_cols, index=train_idx)], axis=1)
X_test_full = pd.concat([X_test_core, pd.DataFrame(test_tfidf.toarray(),
                        columns=tfidf_cols, index=test_idx)], axis=1)
print(f'Training papers: {len(train_idx)}; held-out papers: {len(test_idx)}; TF-IDF terms: {len(tfidf_cols)}')


In [ ]:
# 表格1：回归结果表（Odds Ratio + 显著性）
X_const = sm.add_constant(X_train_core, has_constant="add")
logit_model = sm.Logit(y_train, X_const).fit(disp=0)

reg_table = pd.DataFrame({
    'Variable': X_const.columns,
    'Coef': logit_model.params.round(4),
    'Odds_Ratio': np.exp(logit_model.params).round(4),
    'P_value': logit_model.pvalues.round(4),
    'Significance': ['***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else '' for p in logit_model.pvalues]
})
reg_table.to_csv(OUTPUT_DIR / 'regression_results.csv', index=False)
print("表格1 已生成 → regression_results.csv")
print("\n" + "="*60)
print("LOGISTIC REGRESSION RESULTS (ICLR PRESENTATION-LABEL ASSOCIATIONS)")
print("="*60)
# 使用 to_string() 确保格式对齐
print(reg_table.to_string(index=False))
print("-" * 60)
print("Significance codes: 0 '***' 0.01 '**' 0.05 '*' 0.1")
print(f"Log-Likelihood: {logit_model.llf:.2f} | Pseudo R-squared: {logit_model.prsquared:.4f}")
print("="*60)

In [ ]:
# Table 2. Exploratory nearest-neighbor COVARIATE matching, with replacement.
# No propensity scores are estimated. The matched contrast is not a causal effect.
median_prom = df['promotional_ratio'].median()
higher_promotional = df['promotional_ratio'] > median_prom
covariates = ['max_h_index', 'author_count', 'word_count', 'certainty', 'achievement']
# This descriptive analysis uses the full sample; it does not evaluate holdout prediction.
matching_scaler = StandardScaler()
matching_features = pd.DataFrame(matching_scaler.fit_transform(df[covariates]),
                                columns=covariates, index=df.index)
higher_group = df.loc[higher_promotional].copy()
comparison_group = df.loc[~higher_promotional].copy()
if higher_group.empty or comparison_group.empty:
    raise ValueError('Matching requires nonempty groups above and at/below the promotional-ratio median.')
nn = NearestNeighbors(n_neighbors=1)
nn.fit(matching_features.loc[comparison_group.index])
_, indices = nn.kneighbors(matching_features.loc[higher_group.index])
matched_comparison = comparison_group.iloc[indices.flatten()].copy()
rate_contrast = higher_group['is_high_level'].mean() - matched_comparison['is_high_level'].mean()
matching_table = pd.DataFrame({
    'Group': ['Above-median promotional ratio', 'Matched comparison', 'Descriptive rate contrast'],
    'Oral_Spotlight_Rate': [higher_group['is_high_level'].mean(),
                           matched_comparison['is_high_level'].mean(), rate_contrast],
    'N': [len(higher_group), len(matched_comparison), np.nan]
})
matching_table.to_csv(OUTPUT_DIR / 'matching_results.csv', index=False)
print(matching_table.to_string(index=False))
print('Nearest-neighbor covariate matching (k=1), with replacement; no caliper or balance assessment.')
print('Exploratory association only: unmeasured confounding and selection remain possible.')


In [ ]:
# Table 3. Held-out discrimination; the models use different feature sets.
pseudo_r2 = 1 - (logit_model.llf / logit_model.llnull)
rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
rf.fit(X_train_full, y_train)
logit_prob = logit_model.predict(sm.add_constant(X_test_core, has_constant='add'))
rf_prob = rf.predict_proba(X_test_full)[:, 1]
model_table = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Always Poster'],
    'Feature_Set': ['Numeric core', 'Numeric core + TF-IDF', 'No features'],
    'Test_AUC': [roc_auc_score(y_test, logit_prob), roc_auc_score(y_test, rf_prob), 0.5],
    'Test_Accuracy': [accuracy_score(y_test, logit_prob >= 0.5), rf.score(X_test_full, y_test), (y_test == 0).mean()],
    'Training_Pseudo_R2': [pseudo_r2, np.nan, np.nan]
})
model_table.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
print(model_table.round(4).to_string(index=False))
print('This compares two configurations, not an isolated algorithm effect. No hyperparameter search was performed.')
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(model_table['Model'], model_table['Test_AUC'], color=['teal', 'slateblue', 'gray'])
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_ylabel('Held-out ROC AUC')
ax.set_title('Oral/Spotlight vs Poster: one stratified holdout')
for i, value in enumerate(model_table['Test_AUC']):
    ax.text(i, value + 0.015, f'{value:.3f}', ha='center')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=160)
plt.show()


In [ ]:
# Table 4. Exploratory year-by-year associations (raw numeric units)
robust_list = []
for year in sorted(df['year'].unique()):
    sub = df[df['year'] == year]
    if len(sub) < 30: continue
    X_sub = sm.add_constant(sub[feature_cols])
    y_sub = sub['is_high_level']
    try:
        m = sm.Logit(y_sub, X_sub).fit(disp=0)
        robust_list.append({
            'Year': year,
            'N': len(sub),
            'Promotional_Coef': round(m.params.get('promotional_ratio', np.nan), 4),
            'P_value': round(m.pvalues.get('promotional_ratio', np.nan), 4),
            'Pseudo_R2': round(1 - (m.llf / m.llnull), 4)
        })
    except Exception as exc:
        print(f"Year {year}: fit skipped ({type(exc).__name__}).")

robust_table = pd.DataFrame(robust_list)
robust_table.to_csv(OUTPUT_DIR / 'yearly_associations.csv', index=False)
print("表格4 已生成 → yearly_associations.csv")
print("\n" + "="*60)
print("EXPLORATORY YEAR-BY-YEAR ASSOCIATIONS (PROMOTIONAL RATIO)")
print("="*60)
if not robust_table.empty:
    print(robust_table.to_string(index=False))
else:
    print("Warning: No robustness results generated (insufficient data per year).")
print("-" * 60)
print(f"Goal: Describe how the promotional-ratio association varies by year.")
print(f"Coefficients use raw feature units, unlike the training-standardized main model; no causal or stability conclusion is established.")
print("="*60)

In [ ]:
#5. 变量分布图
dist_stats = []
plt.figure(figsize=(15, 12))

for i, col in enumerate(num_features):
    plt.subplot(3, 3, i+1)

    # 核心改进：计算 99% 分位数，仅用于绘图展示，避免长尾挤压
    upper_limit = df[col].quantile(0.99)
    plot_data = df[df[col] <= upper_limit][col].dropna()

    # 绘制直方图 + KDE
    sns.histplot(plot_data, kde=True, color='teal', bins=30)

    # 计算原始数据的偏度与峰度（保持统计真实性）
    s, k = skew(df[col].dropna()), kurtosis(df[col].dropna())
    dist_stats.append({'Feature': col, 'Mean': df[col].mean().round(2), 'Skew': s, 'Kurt': k})

    # 标题加入修饰说明
    plt.title(f'{col}\n(Shown: <99th Percentile)\nSkew: {s:.2f}, Kurt: {k:.2f}', fontsize=10)
    plt.xlabel('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_distributions.png", dpi=160)
plt.show()

# 输出统计表
print("\n分布统计摘要：")
print(pd.DataFrame(dist_stats).to_string(index=False))

In [ ]:
# =======================
# 6. 跨类别分布差异对比
# 图表：小提琴图 (Violin Plot)
# 目的：对比Oral/Spotlight 与 Poster 论文在核心特征（如宣传比例）上的中位数和概率密度
# =======================

compare_cols = [
    'promotional_ratio', 'max_h_index', 'avg_citations', 'word_count',
    'sent_count', 'author_count', 'pos_score', 'neg_score',
    'certainty', 'achievement'
]
plt.figure(figsize=(15, 20))

for i, col in enumerate(compare_cols):
    plt.subplot(5, 2, i+1)
    sns.violinplot(x='is_high_level', y=col, data=df, hue="is_high_level", inner="quart", palette="Set3", legend=False)
    plt.title(f'Density Comparison: {col}')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 7. 按展示类别比较描述性斜率
# 图表：分类回归线性图 (lmplot)
# 目的：描述作者H-index与语言特征在不同展示类别中的关联
# ==========================================
analyze_list = ['promotional_ratio', 'word_count', 'pos_score']
base_feature = 'max_h_index'  # 保持以“名望”作为描述性横轴

for target_col in analyze_list:
    # --- 1. 绘图部分 ---
    g = sns.lmplot(x=base_feature, y=target_col, hue='is_high_level',
                   data=df, markers=["o", "x"], palette="Set1", height=6, aspect=1.2)

    plt.xlim(0, df[base_feature].quantile(0.99))
    plt.title(f'Descriptive trends: {base_feature} vs {target_col}')

    # --- 2. 统计计算部分 ---
    slopes = {}
    for level in [0, 1]:
        sub = df[df['is_high_level'] == level].dropna(subset=[base_feature, target_col])
        res = sm.OLS(sub[target_col], sm.add_constant(sub[base_feature])).fit()
        slopes[level] = res.params[base_feature]

    print("\n" + "="*60)
    print(f"分析目标: {target_col} 与 {base_feature} 的分组描述性关系")
    print("-" * 60)
    print(f"Group 0 (Poster) Slope: {slopes[0]:.6f}")
    print(f"Group 1 (Oral/Spotlight) Slope: {slopes[1]:.6f}")

    diff = slopes[1] - slopes[0]
    print(f'Descriptive slope difference: {diff:.6f}')
    print('No formal interaction test or causal moderation claim is made.')
    print("="*60)
    plt.show()


In [ ]:
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer

# ==========================================
# 8. NLP 词云图
# ==========================================
high_text = " ".join(df[df['is_high_level']==1]['abstract'].astype(str))
# prefer_horizontal=1.0 确保所有单词水平排列，看起来更整齐
# relative_scaling=0.5 平衡单词频率与字体大小，避免最大词过大
wc = WordCloud(
    width=1000,
    height=600,
    background_color='white',
    colormap='viridis',       # 使用专业配色
    max_words=100,            # 限制单词数量避免混乱
    prefer_horizontal=0.9,    # 90%的单词水平显示
    relative_scaling=0.5,     # 字体大小平滑度
    min_font_size=10,
    random_state=42           # 固定布局
)

# 生成词云
wc.generate(high_text)

plt.figure(figsize=(12, 8))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')  # 隐藏坐标轴
plt.title('Key Themes in Oral/Spotlight Papers', fontsize=16, pad=20)
plt.show()

# ==========================================
# 2. N-gram (Bigrams) 分析
# 目的：展示最常出现的“词组对”，发现学术表达套路
# ==========================================
from sklearn.feature_extraction.text import CountVectorizer

# 只提取前 10 个最频繁的双词组合 (Bigrams)
cv = CountVectorizer(ngram_range=(2, 2), stop_words='english', max_features=10)
bigrams = cv.fit_transform(df['abstract'].astype(str))
counts = pd.DataFrame(bigrams.toarray(), columns=cv.get_feature_names_out()).sum().sort_values()

plt.figure(figsize=(10, 6))
bars = plt.barh(counts.index, counts.values, color='skyblue', edgecolor='navy')

# 在条形图末尾添加数值标签
for bar in bars:
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{int(bar.get_width())}', va='center', fontsize=10)

plt.title('Top 10 Common Phrases (Bigrams) in Abstracts', fontsize=14)
plt.xlabel('Frequency')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# ==========================================
# 9. 全变量相关性矩阵 (Full Correlation Matrix)
# ==========================================
all_analysis_cols = [
    'is_high_level',
    'promotional_ratio', 'certainty', 'achievement',
    'pos_score', 'neg_score',
    'max_h_index', 'avg_citations',
    'word_count', 'sent_count', 'author_count'
]

corr = df[all_analysis_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .8})

# 优化坐标轴：旋转 45 度防止重叠
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.title('Full Correlation Matrix: Text Features & Author Metrics', fontsize=15, pad=20)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_matrix.png", dpi=160)
plt.show()

# --- 自动 print 关键发现 ---
print("\n" + "="*60)
print("CORRELATION ANALYSIS SUMMARY")
print("-" * 60)
# 提取与展示类别(is_high_level)相关性最强的 top 3 变量
top_corr = corr['is_high_level'].sort_values(ascending=False)
print("与展示类别相关性最高（正向）的特征：")
print(top_corr[1:4].to_string(header=False))
print("\n与展示类别相关性最低（负向）的特征：")
print(top_corr.tail(3).to_string(header=False))
print("="*60)

In [ ]:
# ==========================================
# 10. 变量间相关性矩阵图 (Pair Plot)
# 图表说明：对角线展示分布，非对角线展示两两变量的散点关系
# ==========================================
selected_cols = ['promotional_ratio', 'max_h_index', 'avg_citations', 'is_high_level']

g = sns.pairplot(df[selected_cols], hue='is_high_level', diag_kind='kde',
                 palette='husl', plot_kws={'alpha':0.5})
g.fig.suptitle('Pairwise Relationship Matrix by Paper Level', y=1.02, fontsize=16)
plt.show()

print("\n" + "="*60)
print("变量间交互关系矩阵 (Pair Plot) 分析维度：")
print("-" * 60)
print("1. 对角线 (Diagonal): 展示各特征在不同展示类别下的【概率密度分布】。")
print("   - 观察两色曲线的重合度：重合度越低，说明该变量对录取的区分度越高。")
print("\n2. 非对角线 (Off-diagonal): 展示两两变量间的【散点相关性】。")
print("   - 观察是否存在线性趋势，或不同展示类别的点是否分布不同。")
print("\n3. 离群值 (Outliers): 观察极端学术大牛（高引用/高H指数）的分布。")
print("   - 只描述样本中的极端作者指标，不推断未录取论文的表现。")
print("="*60)